
#PARTE 5 - Validación e Inferencia usando FashionMNIST

En esta parte se implementa el proceso de validación e inferencia del modelo
utilizando el dataset FashionMNIST.

Se evalua el comportamiento de la red neuronal sobre datos no vistos durante
el entrenamiento con el fin de verificar su capacidad de generalización y
reducir problemas de sobreajuste (overfitting).

Ademas, se realiza una prediccion final para observar el desempeño del modelo.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

*1 Configuracion*

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo:", device)

Dispositivo: cpu


*3 Carga del dataset*

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = datasets.FashionMNIST(
    'FashionMNIST_data/',
    train=True,
    download=True,
    transform=transform
)

testset = datasets.FashionMNIST(
    'FashionMNIST_data/',
    train=False,
    download=True,
    transform=transform
)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

classes = trainset.classes

*4 Construcción de la CNN*

In [4]:
class CNN_FashionMNIST(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv_layers = nn.Sequential(

            nn.Conv2d(1,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc_layers = nn.Sequential(

            nn.Flatten(),

            nn.Linear(64*7*7,128),
            nn.ReLU(),

            # Dropout para reducir overfitting
            nn.Dropout(0.5),

            nn.Linear(128,10)
        )

    def forward(self,x):

        x = self.conv_layers(x)
        x = self.fc_layers(x)

        return x

*5 Crear modelo*

In [5]:
model = CNN_FashionMNIST().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

*6 Función de validación*

In [6]:
def validation(model, testloader, criterion):

    test_loss = 0
    correct = 0
    total = 0

    model.eval()

    with torch.no_grad():

        for images, labels in testloader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            test_loss += loss.item()

            _, predicted = torch.max(outputs,1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    return test_loss/len(testloader), accuracy

*7 Entrenamiento con validación*

In [7]:
epochs = 10

train_losses = []
test_losses = []
accuracies = []

for epoch in range(epochs):

    running_loss = 0

    model.train()

    for images, labels in trainloader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(trainloader)

    test_loss, accuracy = validation(
        model,
        testloader,
        criterion
    )

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    accuracies.append(accuracy)

    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"Training Loss: {train_loss:.4f}")
    print(f"Validation Loss: {test_loss:.4f}")
    print(f"Accuracy: {accuracy:.2f}%")
    print("-"*40)

Epoch [1/10]
Training Loss: 0.5521
Validation Loss: 0.3566
Accuracy: 87.11%
----------------------------------------
Epoch [2/10]
Training Loss: 0.3665
Validation Loss: 0.3102
Accuracy: 88.81%
----------------------------------------
Epoch [3/10]
Training Loss: 0.3125
Validation Loss: 0.2699
Accuracy: 90.30%
----------------------------------------
Epoch [4/10]
Training Loss: 0.2789
Validation Loss: 0.2584
Accuracy: 90.62%
----------------------------------------
Epoch [5/10]
Training Loss: 0.2535
Validation Loss: 0.2466
Accuracy: 90.95%
----------------------------------------
Epoch [6/10]
Training Loss: 0.2333
Validation Loss: 0.2601
Accuracy: 90.59%
----------------------------------------
Epoch [7/10]
Training Loss: 0.2187
Validation Loss: 0.2425
Accuracy: 91.50%
----------------------------------------
Epoch [8/10]
Training Loss: 0.2055
Validation Loss: 0.2470
Accuracy: 91.55%
----------------------------------------
Epoch [9/10]
Training Loss: 0.1902
Validation Loss: 0.2641
Accur

*8 Graficas*

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(train_losses)
plt.plot(test_losses)
plt.legend(["Train Loss","Validation Loss"])
plt.title("Pérdida del modelo")
plt.show()


plt.figure(figsize=(10,4))
plt.plot(accuracies)
plt.title("Accuracy")
plt.ylabel("%")
plt.show()

: 

*9 Inferencia (predicción final)*

In [ ]:
images, labels = next(iter(testloader))

model.eval()

with torch.no_grad():

    outputs = model(images.to(device))

    _, predicted = torch.max(outputs,1)

*10 Resultado*

In [ ]:
plt.figure(figsize=(4,4))

plt.imshow(images[0].squeeze(), cmap='gray')

plt.title(
    f"Real: {classes[labels[0].item()]} | Predicción: {classes[predicted[0].item()]}"
)

plt.axis("off")

plt.show()

*11 Guardar modelo*

In [ ]:
torch.save(
    model.state_dict(),
    "cnn_fashionmnist_part5.pth"
)

print("Modelo guardado correctamente")